In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_SOURCE_CT = True
REUSE_INPUTS = True
REUSE_CANDIDATES = False
REUSE_TRACES = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — LAD Wide Proximal Reacquisition

Starts from the post-hoc validated ~24.997 mm LAD. Searches a 3–12 mm 3-D shell for compact coronary-sized candidates without using the aorta for discovery or trace steering. Every candidate must have local bidirectional support and must reconnect to an independently traced LAD-side path after dense final-path-tangent QC.


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --depth 1 --branch lad-wide-proximal-reacquisition-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install SimpleITK scipy pandas matplotlib
import sys
sys.path.insert(0,'/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.lad_wide_proximal_reacquisition import LADWideProximalReacquisitionWorkflow, synthetic_wide_reacquisition_self_test
test=synthetic_wide_reacquisition_self_test(); display(test); assert test['passed'], test
wf=LADWideProximalReacquisitionWorkflow(root='/content/drive/MyDrive/OpenPlaque', reuse={'source_ct':REUSE_SOURCE_CT,'inputs':REUSE_INPUTS,'candidates':REUSE_CANDIDATES,'traces':REUSE_TRACES,'figures':REUSE_FIGURES,'report':REUSE_REPORT})
display(wf.cache_status())


In [ ]:
wf.load_source_ct(); prior=wf.load_inputs(); wf.load_aorta_ranker(); display(prior)
print('Validated LAD input length:', wf.summary if wf.summary else prior.get('combined_lad_length_mm'))


In [ ]:
candidates=wf.discover_candidates(); print('Wide candidates:', len(candidates)); display(candidates.head(20))


In [ ]:
summary=wf.run_reconnection(); display(summary)
print('Reconnection results:'); display(wf.candidate_qc)


In [ ]:
names=wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report=wf.build_report(); zip_path=wf.package()
print('STATUS:', wf.summary['status'])
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_WIDE_PROXIMAL_REACQUISITION_REPORT_BACK.zip')
